In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random

In [2]:
conn = sqlite3.connect("./my_database.db")

In [3]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [4]:
test_vendor = vendor_ids.iloc[2]['vendor_id']
test_vendor

'1031b882-b8c6-4750-80ab-9a4767d2da65'

In [5]:
def unassigned_work_orders(vendor_id):
    df = pd.read_sql(
        f"""
        select count(*) as unassigned_work_orders
        from work_orders
        where assigned_vendor='{vendor_id}'"""
    , con=conn)
    return {'vendor_id': vendor_id, 'unassigned work orders': df['unassigned_work_orders'].iloc[0]}

In [6]:
unassigned_work_orders(test_vendor)

{'vendor_id': '1031b882-b8c6-4750-80ab-9a4767d2da65',
 'unassigned work orders': np.int64(98)}

In [7]:
def pending_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and (status = 'in progress' or status = 'assigned')
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [8]:
pending_tickets(test_vendor)

{'vendor_id': '1031b882-b8c6-4750-80ab-9a4767d2da65',
 'active tickets': np.int64(379)}

In [12]:
df = pd.read_sql(
    f"""select * 
    from ticket 
    where vendor_id='{test_vendor}' and completed_at is not null and created_at > date('now', '-9 months')"""
    , con=conn, parse_dates=['assigned_at', 'completed_at'])

In [13]:
df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.start_time)

In [14]:
complete_by_week = df.groupby('completion_week')['ticket_id'].count().reset_index().sort_values(by='ticket_id', ascending=False)
complete_by_week

,completion_week,ticket_id
4,2025-09-15,6
5,2025-09-22,4
7,2025-11-03,2
0,2025-08-18,1
1,2025-08-25,1
3,2025-09-08,1
2,2025-09-01,1
6,2025-09-29,1
8,2025-11-24,1
